# High-Performance GPU Kernel Engineering: Matrix Multiplication on NVIDIA Tesla T4 (sm_75)

Welcome! In this walkthrough, we will optimize General Matrix Multiplication (GEMM) in CUDA C++ from the ground up, specifically targeting the **NVIDIA Tesla T4 (Turing Architecture, Compute Capability sm_75)** available in Google Colab.

### NVIDIA Tesla T4 Hardware Budget:
- **SMs**: 40 Streaming Multiprocessors
- **CUDA Cores**: 2,560 FP32 cores (64 per SM)
- **Tensor Cores**: 320 Turing Tensor Cores (8 per SM)
- **Theoretical Peak FP32 Throughput**: $\approx 8.1$ TFLOPS
- **Theoretical Peak FP16 Tensor Core Throughput**: $\approx 65$ TFLOPS
- **Memory**: 16 GB GDDR6 @ $300\text{--}320$ GB/s memory bandwidth
- **Shared Memory / L1 Pool**: 64 KB configurable per SM (32 banks, 4 bytes/bank)

In [ ]:
# Check that you are assigned an NVIDIA Tesla T4
!nvidia-smi

## 1. Writing the Complete CUDA Source Code

Run the following cell to write `sgemm_t4.cu`. It contains:
1. **Naive SGEMM**: Baseline with 1-thread-per-element.
2. **2D Shared-Memory Tiled SGEMM ($32 \times 32$)**: Coalesced cooperative loading into `__shared__` memory.
3. **Register-Tiled SGEMM ($128 \times 128$ block, $8 \times 8$ per thread)**: Vectorized `float4` loads and register reuse.
4. **Turing Hardware Tensor Cores (`nvcuda::wmma`)**: $16 \times 16 \times 16$ tile hardware acceleration.
5. **Benchmarking Harness**: Warmups, CUDA Event timers (`cudaEventRecord`), and CPU correctness verification.

In [ ]:
%%writefile sgemm_t4.cu
#include <iostream>
#include <iomanip>
#include <vector>
#include <cmath>
#include <cuda_runtime.h>
#include <mma.h>
#include <cuda_fp16.h>

#define CUDA_CHECK(call)                                                      \
    do {                                                                      \
        cudaError_t err = call;                                               \
        if (err != cudaSuccess) {                                             \
            std::cerr << "CUDA Error at " << __FILE__ << ":" << __LINE__      \
                      << " -> " << cudaGetErrorString(err) << std::endl;      \
            exit(EXIT_FAILURE);                                               \
        }                                                                     \
    } while (0)

// 1. NAIVE SGEMM
__global__ void sgemm_naive(int M, int N, int K, const float *A, const float *B, float *C) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < M && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < K; ++k) {
            sum += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = sum;
    }
}

// 2. 2D SHARED-MEMORY TILED SGEMM
template <int TILE_DIM>
__global__ void sgemm_shared_tiled(int M, int N, int K, const float *A, const float *B, float *C) {
    __shared__ float s_A[TILE_DIM][TILE_DIM];
    __shared__ float s_B[TILE_DIM][TILE_DIM];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int row = blockIdx.y * TILE_DIM + ty;
    int col = blockIdx.x * TILE_DIM + tx;

    float sum = 0.0f;
    int numTiles = (K + TILE_DIM - 1) / TILE_DIM;

    for (int t = 0; t < numTiles; ++t) {
        int a_col = t * TILE_DIM + tx;
        int b_row = t * TILE_DIM + ty;

        s_A[ty][tx] = (row < M && a_col < K) ? A[row * K + a_col] : 0.0f;
        s_B[ty][tx] = (b_row < K && col < N) ? B[b_row * N + col] : 0.0f;

        __syncthreads();

        #pragma unroll
        for (int k = 0; k < TILE_DIM; ++k) {
            sum += s_A[ty][k] * s_B[k][tx];
        }

        __syncthreads();
    }

    if (row < M && col < N) {
        C[row * N + col] = sum;
    }
}

// 3. REGISTER-TILED (THREAD-TILED) SGEMM WITH VECTORIZED LOADS
constexpr int BM = 128;
constexpr int BN = 128;
constexpr int BK = 8;
constexpr int TM = 8;
constexpr int TN = 8;

__global__ void sgemm_register_tiled(int M, int N, int K, const float *A, const float *B, float *C) {
    int tid = threadIdx.y * blockDim.x + threadIdx.x;
    __shared__ float s_A[BM][BK];
    __shared__ float s_B[BK][BN];

    int threadCol = threadIdx.x;
    int threadRow = threadIdx.y;

    float regC[TM][TN] = {0.0f};
    float regA[TM];
    float regB[TN];

    int blockRow = blockIdx.y * BM;
    int blockCol = blockIdx.x * BN;

    int loadA_row = tid / 2;
    int loadA_col = (tid % 2) * 4;
    int loadB_row = tid / 32;
    int loadB_col = (tid % 32) * 4;

    for (int bkIdx = 0; bkIdx < K; bkIdx += BK) {
        int currA_row = blockRow + loadA_row;
        int currA_col = bkIdx + loadA_col;
        if (currA_row < M && (currA_col + 3) < K) {
            float4 valA = *reinterpret_cast<const float4*>(&A[currA_row * K + currA_col]);
            s_A[loadA_row][loadA_col + 0] = valA.x;
            s_A[loadA_row][loadA_col + 1] = valA.y;
            s_A[loadA_row][loadA_col + 2] = valA.z;
            s_A[loadA_row][loadA_col + 3] = valA.w;
        } else {
            #pragma unroll
            for (int i = 0; i < 4; ++i) {
                s_A[loadA_row][loadA_col + i] = (currA_row < M && (currA_col + i) < K) ? A[currA_row * K + (currA_col + i)] : 0.0f;
            }
        }

        int currB_row = bkIdx + loadB_row;
        int currB_col = blockCol + loadB_col;
        if (currB_row < K && (currB_col + 3) < N) {
            float4 valB = *reinterpret_cast<const float4*>(&B[currB_row * N + currB_col]);
            s_B[loadB_row][loadB_col + 0] = valB.x;
            s_B[loadB_row][loadB_col + 1] = valB.y;
            s_B[loadB_row][loadB_col + 2] = valB.z;
            s_B[loadB_row][loadB_col + 3] = valB.w;
        } else {
            #pragma unroll
            for (int i = 0; i < 4; ++i) {
                s_B[loadB_row][loadB_col + i] = (currB_row < K && (currB_col + i) < N) ? B[currB_row * N + (currB_col + i)] : 0.0f;
            }
        }

        __syncthreads();

        #pragma unroll
        for (int dotIdx = 0; dotIdx < BK; ++dotIdx) {
            #pragma unroll
            for (int i = 0; i < TM; ++i) regA[i] = s_A[threadRow * TM + i][dotIdx];
            #pragma unroll
            for (int j = 0; j < TN; ++j) regB[j] = s_B[dotIdx][threadCol * TN + j];

            #pragma unroll
            for (int i = 0; i < TM; ++i) {
                #pragma unroll
                for (int j = 0; j < TN; ++j) {
                    regC[i][j] += regA[i] * regB[j];
                }
            }
        }
        __syncthreads();
    }

    #pragma unroll
    for (int i = 0; i < TM; ++i) {
        int r = blockRow + threadRow * TM + i;
        #pragma unroll
        for (int j = 0; j < TN; ++j) {
            int c = blockCol + threadCol * TN + j;
            if (r < M && c < N) C[r * N + c] = regC[i][j];
        }
    }
}

// 4. HARDWARE TENSOR CORE GEMM (nvcuda::wmma)
using namespace nvcuda;
constexpr int WMMA_M = 16;
constexpr int WMMA_N = 16;
constexpr int WMMA_K = 16;

__global__ void wmma_gemm_t4(int M, int N, int K, const half *A, const half *B, float *C) {
    int warpId = (threadIdx.y * blockDim.x + threadIdx.x) / 32;
    int warpRow = warpId / 2;
    int warpCol = warpId % 2;

    int cRow = blockIdx.y * 32 + warpRow * WMMA_M;
    int cCol = blockIdx.x * 32 + warpCol * WMMA_N;

    wmma::fragment<wmma::matrix_a, WMMA_M, WMMA_N, WMMA_K, half, wmma::row_major> a_frag;
    wmma::fragment<wmma::matrix_b, WMMA_M, WMMA_N, WMMA_K, half, wmma::row_major> b_frag;
    wmma::fragment<wmma::accumulator, WMMA_M, WMMA_N, WMMA_K, float> c_frag;

    wmma::fill_fragment(c_frag, 0.0f);

    for (int k = 0; k < K; k += WMMA_K) {
        if (cRow < M && k < K) {
            wmma::load_matrix_sync(a_frag, A + cRow * K + k, K);
            wmma::load_matrix_sync(b_frag, B + k * N + cCol, N);
            wmma::mma_sync(c_frag, a_frag, b_frag, c_frag);
        }
    }

    if (cRow < M && cCol < N) {
        wmma::store_matrix_sync(C + cRow * N + cCol, c_frag, N, wmma::mem_row_major);
    }
}

void cpu_gemm(int M, int N, int K, const float *A, const float *B, float *C) {
    for (int i = 0; i < M; ++i) {
        for (int k = 0; k < K; ++k) {
            float a = A[i * K + k];
            for (int j = 0; j < N; ++j) C[i * N + j] += a * B[k * N + j];
        }
    }
}

bool verify_matrix(int M, int N, const float *expected, const float *actual, float tol = 1e-3f) {
    float max_diff = 0.0f;
    for (int i = 0; i < M * N; ++i) {
        float diff = std::abs(expected[i] - actual[i]);
        if (diff > max_diff) max_diff = diff;
        if (diff > tol) {
            std::cout << "Mismatch at " << i << ": exp=" << expected[i] << ", act=" << actual[i] << std::endl;
            return false;
        }
    }
    std::cout << "Verification PASSED (Max abs diff: " << max_diff << ")" << std::endl;
    return true;
}

int main() {
    const int M = 2048, N = 2048, K = 2048;
    size_t bytes_f = M * N * sizeof(float);
    std::vector<float> h_A(M * K), h_B(K * N), h_C(M * N, 0.0f), h_C_ref(M * N, 0.0f);
    for (int i = 0; i < M * K; ++i) h_A[i] = static_cast<float>(rand()) / RAND_MAX - 0.5f;
    for (int i = 0; i < K * N; ++i) h_B[i] = static_cast<float>(rand()) / RAND_MAX - 0.5f;

    const int VERIF_DIM = 512;
    cpu_gemm(VERIF_DIM, VERIF_DIM, VERIF_DIM, h_A.data(), h_B.data(), h_C_ref.data());

    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, bytes_f));
    CUDA_CHECK(cudaMalloc(&d_B, bytes_f));
    CUDA_CHECK(cudaMalloc(&d_C, bytes_f));
    CUDA_CHECK(cudaMemcpy(d_A, h_A.data(), bytes_f, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B.data(), bytes_f, cudaMemcpyHostToDevice));

    half *d_A_half, *d_B_half;
    std::vector<half> h_A_half(M * K), h_B_half(K * N);
    for (int i = 0; i < M * K; ++i) h_A_half[i] = __float2half(h_A[i]);
    for (int i = 0; i < K * N; ++i) h_B_half[i] = __float2half(h_B[i]);
    CUDA_CHECK(cudaMalloc(&d_A_half, M * K * sizeof(half)));
    CUDA_CHECK(cudaMalloc(&d_B_half, K * N * sizeof(half)));
    CUDA_CHECK(cudaMemcpy(d_A_half, h_A_half.data(), M * K * sizeof(half), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B_half, h_B_half.data(), K * N * sizeof(half), cudaMemcpyHostToDevice));

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    const int WARMUP = 5, BENCH = 20;
    double total_flops = 2.0 * static_cast<double>(M) * N * K;

    auto bench = [&](const char *name, auto func, bool is_wmma = false) {
        for (int i = 0; i < WARMUP; ++i) func();
        CUDA_CHECK(cudaDeviceSynchronize());
        CUDA_CHECK(cudaEventRecord(start));
        for (int i = 0; i < BENCH; ++i) func();
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
        float avg_ms = ms / BENCH;
        double tflops = (total_flops * 1e-12) / (avg_ms * 1e-3);
        double gflops = tflops * 1000.0;
        std::cout << std::left << std::setw(32) << name 
                  << " | Time: " << std::fixed << std::setprecision(3) << std::setw(8) << avg_ms << " ms"
                  << " | Throughput: " << std::setw(6) << std::setprecision(2) << tflops << " TFLOPS"
                  << " (" << std::setw(7) << std::setprecision(1) << gflops << " GFLOPS)" << std::endl;
        CUDA_CHECK(cudaMemcpy(h_C.data(), d_C, bytes_f, cudaMemcpyDeviceToHost));
        verify_matrix(VERIF_DIM, VERIF_DIM, h_C_ref.data(), h_C.data(), is_wmma ? 1e-1f : 1e-3f);
        std::cout << "-----------------------------------------------------------------------------------\n";
    };

    std::cout << "Benchmarking M=N=K=2048 on Tesla T4 (sm_75):\n";
    std::cout << "-----------------------------------------------------------------------------------\n";
    { dim3 b(32, 32); dim3 g((N + 31)/32, (M + 31)/32); bench("1. Naive SGEMM", [&](){ sgemm_naive<<<g, b>>>(M, N, K, d_A, d_B, d_C); }); }
    { dim3 b(32, 32); dim3 g((N + 31)/32, (M + 31)/32); bench("2. Shared Memory Tiled SGEMM", [&](){ sgemm_shared_tiled<32><<<g, b>>>(M, N, K, d_A, d_B, d_C); }); }
    { dim3 b(16, 16); dim3 g((N + 127)/128, (M + 127)/128); bench("3. Register-Tiled SGEMM (8x8)", [&](){ sgemm_register_tiled<<<g, b>>>(M, N, K, d_A, d_B, d_C); }); }
    { dim3 b(32, 4); dim3 g((N + 31)/32, (M + 31)/32); bench("4. Turing Tensor Cores (WMMA)", [&](){ wmma_gemm_t4<<<g, b>>>(M, N, K, d_A_half, d_B_half, d_C); }, true); }

    cudaEventDestroy(start); cudaEventDestroy(stop);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C); cudaFree(d_A_half); cudaFree(d_B_half);
    return 0;
}


## 2. Compile with `nvcc -arch=sm_75 -O3` and Execute

In [ ]:
!nvcc -arch=sm_75 -O3 sgemm_t4.cu -o sgemm_t4
!./sgemm_t4